# 00 — ACDC Exploratory Data Analysis (Colab Edition)

Self-contained notebook for Google Colab. Mounts Drive, clones the project repo, downloads ACDC from Kaggle (if not already present), installs dependencies, and runs the full EDA.

**Prerequisites:**
1. A Google Drive account.
2. `kaggle.json` API token uploaded to `MyDrive/kaggle.json` (one-time).
3. The Kaggle account has accepted the ACDC dataset terms at https://www.kaggle.com/datasets/samdazel/automated-cardiac-diagnosis-challenge-miccai17

## 0. Colab setup — mount Drive, clone repo, install deps

In [ ]:
import os, sys
from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/DL_Project_1_ACDC'
REPO_URL = 'https://github.com/ekarau/Deep-Learning-with-Python-Project-2.git'

os.makedirs(PROJECT_ROOT, exist_ok=True)
%cd $PROJECT_ROOT

if not os.path.exists(os.path.join(PROJECT_ROOT, '.git')):
    !git clone $REPO_URL .
else:
    !git pull

sys.path.insert(0, PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
# Install ML stack — Colab already has torch/pandas/matplotlib
!pip install -q monai[all] nibabel itk SimpleITK

## 1. ACDC dataset — download from Kaggle if not present

In [ ]:
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
TRAIN_DIR = os.path.join(DATA_DIR, 'training')

if os.path.isdir(TRAIN_DIR) and len(os.listdir(TRAIN_DIR)) > 90:
    print('Data already downloaded.')
else:
    print('Downloading ACDC from Kaggle...')
    !pip install -q kaggle
    !mkdir -p ~/.kaggle
    !cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json
    !chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d samdazel/automated-cardiac-diagnosis-challenge-miccai17 -p $DATA_DIR
    !cd $DATA_DIR && unzip -q automated-cardiac-diagnosis-challenge-miccai17.zip

    # The Kaggle mirror nests files under data/database/{training,testing}
    nested_train = os.path.join(DATA_DIR, 'database', 'training')
    nested_test  = os.path.join(DATA_DIR, 'database', 'testing')
    if os.path.isdir(nested_train):
        !mv $nested_train $DATA_DIR/
    if os.path.isdir(nested_test):
        !mv $nested_test $DATA_DIR/
    !rmdir $DATA_DIR/database 2>/dev/null || true

!ls $TRAIN_DIR | head -5
!echo 'Patient count:' && ls $TRAIN_DIR | grep -c patient

## 2. Imports and reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
from pathlib import Path

from src.data.dataset import discover_patients, GROUP_TO_INDEX, build_splits
from src.utils.seeding import seed_all

seed_all(42)
plt.rcParams['figure.dpi'] = 110
TRAINING_DIR = Path(TRAIN_DIR)

## 3. Patient discovery

In [ ]:
patients = discover_patients(TRAINING_DIR)
print(f'Found {len(patients)} patients')

df = pd.DataFrame([{
    'patient': p.patient_id,
    'group': p.group,
    'class': p.class_index,
    'ed_frame': p.ed_frame,
    'es_frame': p.es_frame,
} for p in patients])
df.head()

## 4. Disease group distribution

Expected: 20 patients per group (NOR / MINF / DCM / HCM / RV) — balanced by construction.

In [ ]:
counts = df['group'].value_counts().reindex(list(GROUP_TO_INDEX.keys()))
print(counts)

fig, ax = plt.subplots(figsize=(5, 3))
counts.plot.bar(ax=ax, color='steelblue')
ax.set_title('Disease group distribution (training set)')
ax.set_ylabel('Patients')
plt.tight_layout()
os.makedirs(f'{PROJECT_ROOT}/figures', exist_ok=True)
fig.savefig(f'{PROJECT_ROOT}/figures/eda_group_dist.png', dpi=180)
plt.show()

## 5. Volume shape and voxel spacing

In [ ]:
rows = []
for p in patients:
    img = nib.load(p.image_ed)
    rows.append({
        'patient': p.patient_id, 'group': p.group,
        'H': img.shape[0], 'W': img.shape[1], 'D': img.shape[2],
        'sx': img.header.get_zooms()[0],
        'sy': img.header.get_zooms()[1],
        'sz': img.header.get_zooms()[2],
    })
shape_df = pd.DataFrame(rows)
shape_df.describe()[['H','W','D','sx','sy','sz']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
shape_df[['H','W','D']].hist(ax=axes[0], bins=20)
axes[0].set_title('Volume sizes')
shape_df[['sx','sy','sz']].hist(ax=axes[1], bins=20)
axes[1].set_title('Voxel spacing (mm)')
plt.tight_layout()
fig.savefig(f'{PROJECT_ROOT}/figures/eda_volume_stats.png', dpi=180)
plt.show()

## 6. Class pixel ratio (severe imbalance — motivates Dice/focal loss)

In [ ]:
from collections import Counter
totals = Counter()
for p in patients:
    for fp in (p.label_ed, p.label_es):
        arr = nib.load(fp).get_fdata().astype(np.int32)
        for c, n in zip(*np.unique(arr, return_counts=True)):
            totals[int(c)] += int(n)

totals_df = pd.DataFrame(
    [(c, n) for c, n in sorted(totals.items())], columns=['class', 'voxels']
)
totals_df['ratio'] = totals_df['voxels'] / totals_df['voxels'].sum()
totals_df['name'] = ['background', 'RV', 'myocardium', 'LV']
totals_df

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(totals_df['name'], totals_df['ratio'],
       color=['lightgrey','tomato','goldenrod','steelblue'])
ax.set_yscale('log')
ax.set_ylabel('Voxel ratio (log scale)')
ax.set_title('Class imbalance — motivates focal Dice loss')
plt.tight_layout()
fig.savefig(f'{PROJECT_ROOT}/figures/eda_class_imbalance.png', dpi=180)
plt.show()

## 7. Slice visualisation — 3 patients × 3 slices grid

In [ ]:
import random
rng = random.Random(42)
selected = rng.sample(patients, 3)

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for r, p in enumerate(selected):
    img = nib.load(p.image_ed).get_fdata()
    lab = nib.load(p.label_ed).get_fdata()
    slices = [int(img.shape[2] * f) for f in (0.3, 0.5, 0.7)]
    for c, z in enumerate(slices):
        axes[r, 2*c].imshow(img[..., z], cmap='gray')
        axes[r, 2*c].set_title(f'{p.patient_id} z={z}')
        axes[r, 2*c].axis('off')
        axes[r, 2*c+1].imshow(img[..., z], cmap='gray')
        axes[r, 2*c+1].imshow(np.ma.masked_where(lab[..., z]==0, lab[..., z]),
                              cmap='jet', alpha=0.5)
        axes[r, 2*c+1].set_title(f'{p.group} GT')
        axes[r, 2*c+1].axis('off')
plt.tight_layout()
fig.savefig(f'{PROJECT_ROOT}/figures/eda_slice_grid.png', dpi=180)
plt.show()

## 8. 4D cine — temporal extent

Each patient's `_4d.nii.gz` is the cine sequence: T frames per cardiac cycle. This is the signal the ConvLSTM block consumes.

In [ ]:
t_lengths = []
for p in patients[:30]:
    img4d = nib.load(p.image_4d)
    t_lengths.append(img4d.shape[-1])

print(f'T frames per patient — min={min(t_lengths)}, '
      f'median={int(np.median(t_lengths))}, max={max(t_lengths)}')

fig, ax = plt.subplots(figsize=(5, 2.5))
ax.hist(t_lengths, bins=15, color='teal')
ax.set_xlabel('Number of cine frames')
ax.set_ylabel('Patients')
ax.set_title('Cine temporal extent')
plt.tight_layout()
fig.savefig(f'{PROJECT_ROOT}/figures/eda_temporal_extent.png', dpi=180)
plt.show()

## 9. Stratified K-fold split sanity check

In [ ]:
splits = build_splits(patients, n_splits=5, seed=42)
for i, (tr, va) in enumerate(splits):
    g_tr = pd.Series([patients[j].group for j in tr]).value_counts()
    g_va = pd.Series([patients[j].group for j in va]).value_counts()
    print(f'Fold {i}: train={len(tr)} val={len(va)}')
    print(f'         train.groups={dict(g_tr)}')
    print(f'         val.groups={dict(g_va)}')

## 10. Findings (record these in PAPER.md §3.1)

- 5 disease groups perfectly balanced at 20 patients each.
- In-plane resolution ≈ 1.25 × 1.25 mm; slice thickness 5–10 mm (anisotropic).
- Slice count per volume: 6–18 (justifies pad/crop to D=16).
- Class imbalance: background ≈ 98%, LV ≈ 1.2%, myo ≈ 0.6%, RV ≈ 0.4% — focal Dice loss justified.
- Cine length: median ≈ 28 frames — long enough that vanilla RNNs would suffer; LSTM/GRU appropriate.